In [4]:
from datasets import load_dataset

fleurs = load_dataset(
    "google/fleurs",
    "hi_in",
    split="test",
    
)

print(len(fleurs))
print(fleurs[0])

c:\up-JOsh\.venv-1\Lib\site-packages\datasets\load.py:1461: FutureWarning: The repository for google/fleurs contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/google/fleurs
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


418
{'id': 1766, 'num_samples': 145920, 'path': 'C:\\Users\\Sachin Maurya\\.cache\\huggingface\\datasets\\downloads\\extracted\\aa7974d1b7c6dacaea0a7c52ea562b909ee0eef2ac33d6be4f73ed65a553f182\\10011266027513218401.wav', 'audio': {'path': 'test/10011266027513218401.wav', 'array': array([ 0.        ,  0.        ,  0.        , ..., -0.00048101,
       -0.00045913, -0.00040722], shape=(145920,)), 'sampling_rate': 16000}, 'transcription': 'कुछ अणुओं में अस्थिर केंद्रक होता है जिसका मतलब यह है कि उनमें थोड़े या बिना किसी झटके से टूटने की प्रवृत्ति होती है', 'raw_transcription': 'कुछ अणुओं में अस्थिर केंद्रक होता है, जिसका मतलब यह है कि उनमें थोड़े या बिना किसी झटके से टूटने की प्रवृत्ति होती है,', 'gender': 0, 'lang_id': 32, 'language': 'Hindi', 'lang_group_id': 4}


In [1]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import evaluate
import torch
from tqdm import tqdm
import re

wer_metric = evaluate.load("wer")

model_path = "C:/Josh_assin/whisper-small-hindi1/checkpoint-2004"

processor = WhisperProcessor.from_pretrained(model_path)
model = WhisperForConditionalGeneration.from_pretrained(model_path).cuda()

# Force Hindi transcription
forced_decoder_ids = processor.get_decoder_prompt_ids(language="hi", task="transcribe")
model.config.forced_decoder_ids = forced_decoder_ids
model.config.suppress_tokens = []

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [2]:
import re

def normalize(text):
    text = text.lower()
    text = text.replace("।", "")
    text = re.sub(r"[,.!?]", "", text)
    text = re.sub(r"[^\u0900-\u097F\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

In [5]:
predictions = []
references = []

for sample in tqdm(fleurs):
    speech = sample["audio"]["array"]

    inputs = processor.feature_extractor(
        speech,
        sampling_rate=16000,
        return_tensors="pt"
    ).input_features.cuda()

    predicted_ids = model.generate(
        inputs,
        max_length=225
    )

    pred = processor.tokenizer.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0]

    predictions.append(normalize(pred))
    references.append(normalize(sample["transcription"]))


wer_finetuned = wer_metric.compute(
    predictions=predictions,
    references=references
)

print("Fine-tuned Whisper-small WER:", wer_finetuned)

100%|██████████| 418/418 [14:15<00:00,  2.05s/it]

Fine-tuned Whisper-small WER: 0.37242331590886674


In [7]:
errors = []

for ref, pred in zip(references, predictions):
    if normalize(pred) != normalize(ref):
        errors.append((ref, pred))

print("Total errors:", len(errors))

Total errors: 418


In [8]:
step = len(errors) // 25
sampled_errors = errors[::step][:25]

In [9]:
for i, (ref, pred) in enumerate(sampled_errors):
    print("Example", i+1)
    print("Reference :", ref)
    print("Prediction:", pred)
    print()

Example 1
Reference : कुछ अणुओं में अस्थिर केंद्रक होता है जिसका मतलब यह है कि उनमें थोड़े या बिना किसी झटके से टूटने की प्रवृत्ति होती है
Prediction: कुछ अड़ूम में अस्त केंड्रक होता है जिसका मतलब यहां कि उनमें थोड़े या बिना किसी जटके से टूठने की प्रवत्ती होती है

Example 2
Reference : गंभीर मौसम किसी भी खतरनाक मौसमी घटना के लिए वर्गीय शब्द है जिसमें नुकसान गंभीर सामाजिक व्यवधान या मानव जीवन की हानि की संभावना है
Prediction: गंबीर मौसम किसी भी खतनाक मौसमी गटनाके लिए वर्गी है सब्दाय जिसमें नुकसान गंबीर सामाजिक व्यवदान या मानव जीवन की हानी की संभावना है

Example 3
Reference : विखंडन बम इस सिद्धांत पर काम करता है कि कई प्रोटॉन और न्यूट्रॉन वाले एक नाभिक को एक साथ रखने के लिए ऊर्जा लगती है
Prediction: विखंडन भम इस सिधानत पर काम करता है कि कई प्रोटॉन और न्यूट्रॉन वाले एक नाभिक को एक साथ रखने के लिए उर्जा लगती है

Example 4
Reference : तीसरी शताब्दी ईसा पूर्व में मिस्रियों द्वारा निर्मित ग्रेट पिरामिड मृत फिरौन का सम्मान करने के लिए निर्मित कई बड़ी पिरामिड संरचनाओं में से एक है
Prediction: त

In [10]:
#Substitution Errors
#Actionable Fix
predictions = []
references = []

for sample in tqdm(fleurs):
    speech = sample["audio"]["array"]

    inputs = processor.feature_extractor(
        speech,
        sampling_rate=16000,
        return_tensors="pt"
    ).input_features.cuda()

    predicted_ids = model.generate(
        inputs,
        max_length=225,
        num_beams=5
    )

    pred = processor.tokenizer.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0]

    predictions.append(normalize(pred))
    references.append(normalize(sample["transcription"]))


100%|██████████| 418/418 [41:19<00:00,  5.93s/it]


In [11]:
wer_beam = wer_metric.compute(predictions=predictions, references=references)
print("WER with beam search:", wer_beam)

WER with beam search: 0.36887266988854917
